# Exercise 5 (Spark) [5 points]
---

For this exercise, you will work on this JupyterLab notebook, and solve the tasks listed herein. These tasks, in addition to writing Spark code, require you to analyse various query plans and to reason about them.

To get a deeper understanding, and look up the types and definitions of various functions, we recommend that you visit the Spark and Spark SQL documentation.

### a) From SQL literal syntax to Dataframe (and back again)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

# Create Spark session
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("StackExchangeAnalysis")
    .getOrCreate()
)

# Define schemas
comments_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("postid", IntegerType(), True),
    StructField("score", IntegerType(), True),
    StructField("text", StringType(), True),
    StructField("userdisplayname", StringType(), True),
    StructField("userid", IntegerType(), True)
])

posts_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("acceptedanswerid", StringType(), True),
    StructField("answercount", IntegerType(), True),
    StructField("body", StringType(), True),
    StructField("closeddate", TimestampType(), True),
    StructField("commentcount", IntegerType(), True),
    StructField("communityowneddate", TimestampType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("favoritecount", IntegerType(), True),
    StructField("lastactivitydate", TimestampType(), True),
    StructField("lasteditdate", TimestampType(), True),
    StructField("lasteditordisplayname", StringType(), True),
    StructField("lasteditoruserid", StringType(), True),
    StructField("ownerdisplayname", StringType(), True),
    StructField("owneruserid", IntegerType(), True),
    StructField("parentid", IntegerType(), True),
    StructField("posttypeid", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("tags", StringType(), True),
    StructField("title", StringType(), True),
    StructField("viewcount", IntegerType(), True)
])

users_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("aboutme", StringType(), True),
    StructField("accountid", StringType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("displayname", StringType(), True),
    StructField("downvotes", IntegerType(), True),
    StructField("lastaccessdate", TimestampType(), True),
    StructField("location", StringType(), True),
    StructField("profileimageurl", StringType(), True),
    StructField("reputation", IntegerType(), True),
    StructField("upvotes", IntegerType(), True),
    StructField("views", IntegerType(), True),
    StructField("websiteurl", StringType(), True)
])

votes_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("bountyamount", IntegerType(), True),
    StructField("creationdate", TimestampType(), True),
    StructField("postid", IntegerType(), True),
    StructField("userid", IntegerType(), True),
    StructField("votetypeid", IntegerType(), True)
])

# # Load CSV files
# badgesDF = spark.read.options(header=True, inferSchema=True).csv("/home/adbs_shared/stackexchange/badges.csv")
# commentsDF = spark.read.options(header=True, inferSchema=True, multiLine=True).schema(comments_schema).csv("/home/adbs_shared/stackexchange/comments.csv")
# postsDF = spark.read.options(header=True, inferSchema=True, multiLine=True).schema(posts_schema).csv("/home/adbs_shared/stackexchange/posts.csv")
# postlinksDF = spark.read.options(header=True, inferSchema=True).csv("/home/adbs_shared/stackexchange/postlinks.csv")
# usersDF = spark.read.options(header=True, inferSchema=True).schema(users_schema).csv("/home/adbs_shared/stackexchange/users.csv")
# votesDF = spark.read.options(header=True, inferSchema=True).schema(votes_schema).csv("/home/adbs_shared/stackexchange/votes.csv")


# Load CSV files locally
badgesDF = spark.read.options(header=True, inferSchema=True).csv("/home/cam-le/Desktop/DS - Advanced Database System/Exercise_2/JupyterLab_Notebooks/ex5_stackexchange/badges.csv")
commentsDF = spark.read.options(header=True, inferSchema=True, multiLine=True).schema(comments_schema).csv("/home/cam-le/Desktop/DS - Advanced Database System/Exercise_2/JupyterLab_Notebooks/ex5_stackexchange/comments.csv")
postsDF = spark.read.options(header=True, inferSchema=True, multiLine=True).schema(posts_schema).csv("/home/cam-le/Desktop/DS - Advanced Database System/Exercise_2/JupyterLab_Notebooks/ex5_stackexchange/posts.csv")
postlinksDF = spark.read.options(header=True, inferSchema=True).csv("/home/cam-le/Desktop/DS - Advanced Database System/Exercise_2/JupyterLab_Notebooks/ex5_stackexchange/postlinks.csv")
usersDF = spark.read.options(header=True, inferSchema=True).schema(users_schema).csv("/home/cam-le/Desktop/DS - Advanced Database System/Exercise_2/JupyterLab_Notebooks/ex5_stackexchange/users.csv")
votesDF = spark.read.options(header=True, inferSchema=True).schema(votes_schema).csv("/home/cam-le/Desktop/DS - Advanced Database System/Exercise_2/JupyterLab_Notebooks/ex5_stackexchange/votes.csv")

# Create temporary views for Spark SQL
badgesDF.createOrReplaceTempView("badges")
commentsDF.createOrReplaceTempView("comments")
postsDF.createOrReplaceTempView("posts")
postlinksDF.createOrReplaceTempView("postlinks")
usersDF.createOrReplaceTempView("users")
votesDF.createOrReplaceTempView("votes")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/09 22:54:58 WARN Utils: Your hostname, cam-le, resolves to a loopback address: 127.0.1.1; using 10.10.5.207 instead (on interface wlo1)
26/05/09 22:54:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 22:54:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
badgesDF.printSchema()
commentsDF.printSchema()
postsDF.printSchema()
postlinksDF.printSchema()
usersDF.printSchema()
votesDF.printSchema()

root
 |-- id: integer (nullable = true)
 |-- class: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- name: string (nullable = true)
 |-- tagbased: boolean (nullable = true)
 |-- userid: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- postid: integer (nullable = true)
 |-- score: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- userdisplayname: string (nullable = true)
 |-- userid: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- acceptedanswerid: string (nullable = true)
 |-- answercount: integer (nullable = true)
 |-- body: string (nullable = true)
 |-- closeddate: timestamp (nullable = true)
 |-- commentcount: integer (nullable = true)
 |-- communityowneddate: timestamp (nullable = true)
 |-- creationdate: timestamp (nullable = true)
 |-- favoritecount: integer (nullable = true)
 |-- lastactivitydate: timestamp (nullable = true)
 |-- lasteditdate: times

#### Query 1: SQL Literal Syntax --> Dataframe API

In [3]:
query = """
WITH answers AS (
  SELECT p.owneruserid AS user_id, COUNT(v.postid) AS accepted_votes
  FROM posts p
  JOIN votes v ON p.id = v.postid
  WHERE p.posttypeid = '2' AND v.votetypeid = '1'
  GROUP BY p.owneruserid
),
questions AS (
  SELECT p.owneruserid AS user_id, COUNT(v.postid) AS upvotes
  FROM posts p
  JOIN votes v ON p.id = v.postid
  WHERE p.posttypeid = '1' AND v.votetypeid = '2'
  GROUP BY p.owneruserid
)
SELECT a.user_id, a.accepted_votes, q.upvotes
FROM answers a
JOIN questions q ON a.user_id = q.user_id
WHERE a.accepted_votes > q.upvotes;
"""

query1 = spark.sql(query)
query1.show()
# query1.explain()

+-------+--------------+-------+
|user_id|accepted_votes|upvotes|
+-------+--------------+-------+
|   7828|            19|      9|
|  28746|            11|      1|
|   8013|            18|      2|
|    686|            34|     11|
|  38185|             3|      1|
|  95569|             4|      3|
| 111259|             4|      1|
|  28732|             3|      1|
|  66341|             3|      1|
| 116195|             4|      2|
|  99274|             2|      1|
|  11887|             6|      1|
+-------+--------------+-------+



In [4]:

# Query 1: SQL --> DataFrame API equivalent

answers = (
    postsDF.filter(col("posttypeid") == "2")
    .join(
        votesDF.filter(col("votetypeid") == 1),
        postsDF.id == votesDF.postid
    )
    .groupBy(postsDF["owneruserid"].alias("user_id"))
    .agg(count(votesDF["postid"]).alias("accepted_votes"))
)

questions = (
    postsDF.filter(col("posttypeid") == "1")
    .join(
        votesDF.filter(col("votetypeid") == 2),
        postsDF.id == votesDF.postid
    )
    .groupBy(postsDF["owneruserid"].alias("user_id"))
    .agg(count(votesDF["postid"]).alias("upvotes"))
)

# Final join + filter: users where accepted_votes > upvotes
query1_df = (
    answers.join(questions, "user_id")
    .filter(col("accepted_votes") > col("upvotes"))
    .select("user_id", "accepted_votes", "upvotes")
)
query1_df.show()


+-------+--------------+-------+
|user_id|accepted_votes|upvotes|
+-------+--------------+-------+
|   7828|            19|      9|
|  28746|            11|      1|
|   8013|            18|      2|
|    686|            34|     11|
|  38185|             3|      1|
|  95569|             4|      3|
| 111259|             4|      1|
|  28732|             3|      1|
|  66341|             3|      1|
| 116195|             4|      2|
|  99274|             2|      1|
|  11887|             6|      1|
+-------+--------------+-------+



In [5]:
# Check if the results from both queries are the same
query1_sql_results = query1.collect()
query1_df_results = query1_df.collect()
assert set(query1_sql_results) == set(query1_df_results), "Results do not match"

#### Query 2: Dataframe API --> SQL Literal Syntax  

In [6]:
user_stats = postsDF.filter(col("owneruserid").isNotNull()).groupBy("owneruserid").agg(
                      count("*").alias("post_count"),
                      sum("score").alias("total_score")
                  )

filtered = user_stats.filter(col("total_score").isNotNull() & 
                             (col("post_count") >= 50))

result = filtered.withColumn(
    "efficiency",
    col("total_score") / col("post_count")
).orderBy(col("efficiency").desc()).limit(50)

result.show()
#result.explain()

+-----------+----------+-----------+-------------------+
|owneruserid|post_count|total_score|         efficiency|
+-----------+----------+-----------+-------------------+
|       4253|        52|        362|  6.961538461538462|
|      14076|        51|        266|  5.215686274509804|
|      53690|        51|        176|  3.450980392156863|
|          3|       101|        345| 3.4158415841584158|
|          5|        58|        191|  3.293103448275862|
|        805|       114|        341|  2.991228070175439|
|      11032|        98|        263|  2.683673469387755|
|        686|       159|        396|  2.490566037735849|
|          2|        69|        141| 2.0434782608695654|
|      36041|        54|        110|  2.037037037037037|
|          0|       323|        511| 1.5820433436532508|
|       7828|        67|         79| 1.1791044776119404|
|      11887|        69|         81|  1.173913043478261|
|          1|       209|        231|  1.105263157894737|
|          4|       329|       

In [7]:

# Query 2: DataFrame API --> SQL Literal Syntax equivalent

query2_sql = """
WITH user_stats AS (
    SELECT
        owneruserid,
        COUNT(*) AS post_count,
        SUM(score) AS total_score
    FROM posts
    WHERE owneruserid IS NOT NULL
    GROUP BY owneruserid
)

, filtered AS (
    SELECT *
    FROM user_stats
    WHERE total_score IS NOT NULL AND post_count >= 50
)

SELECT
    owneruserid,
    post_count,
    total_score,
    total_score / post_count AS efficiency
FROM filtered
ORDER BY efficiency DESC
LIMIT 50;
"""

query2 = spark.sql(query2_sql)
query2.show()


+-----------+----------+-----------+-------------------+
|owneruserid|post_count|total_score|         efficiency|
+-----------+----------+-----------+-------------------+
|       4253|        52|        362|  6.961538461538462|
|      14076|        51|        266|  5.215686274509804|
|      53690|        51|        176|  3.450980392156863|
|          3|       101|        345| 3.4158415841584158|
|          5|        58|        191|  3.293103448275862|
|        805|       114|        341|  2.991228070175439|
|      11032|        98|        263|  2.683673469387755|
|        686|       159|        396|  2.490566037735849|
|          2|        69|        141| 2.0434782608695654|
|      36041|        54|        110|  2.037037037037037|
|          0|       323|        511| 1.5820433436532508|
|       7828|        67|         79| 1.1791044776119404|
|      11887|        69|         81|  1.173913043478261|
|          1|       209|        231|  1.105263157894737|
|          4|       329|       

In [8]:
# Check if the results from both queries are the same
query2_sql_results = query2.collect()
query2_df_results = result.collect()
assert set(query2_sql_results) == set(query2_df_results), "Results do not match"

### b) SPARK processing model
You are given two independent Spark applications. Each application consists of a sequence of Spark jobs, and each job consists of multiple transformations followed by actions. Two optimization blocks have been introduced in an attempt to improve the performance of these applications.<br>
Based on the provided programs, answer the following questions:
* Ignoring the two optimization blocks, identify the stages and their boundaries for each Spark job: *user_scores*, *tag_scores*, and *user_comment_scores*.
Clearly explain your reasoning using Spark’s processing model.
* For Application 1, analyze and discuss the impact of the two optimization blocks on the performance of the application. Explain whether they improve performance, and justify your answer.
 * For Application 2, analyze and discuss the impact of the two optimization blocks on the performance of the application. Again, explain whether they improve performance, providing justification.


In [ ]:
posts = postsDF.select('id','owneruserid','tags','score').rdd 
posts = posts\
    .filter(lambda x: (x['score'] != None) & (x['owneruserid'] != None) & (x['tags'] != None))\
    .map(lambda x: (x['id'], (x['owneruserid'], x['tags'], int(x['score']))))# (post_id, (user_id, tag, score))

votes = votesDF.rdd
votes = votes.map(lambda x: (x['postid'], 1)) # (post_id, vote_count)

comments = commentsDF.rdd
comments = comments.map(lambda x: (x['postid'], 1)) # (post_id, comment_count)

high_score_posts = posts.filter(lambda x: x[1][2] >= 5)

# ---------------------------
# OPTIMIZATION BLOCK 1
# ---------------------------
high_score_posts = high_score_posts.partitionBy(4).cache()
votes = votes.partitionBy(4)
#-----------------------------

post_votes = high_score_posts.join(votes)
post_comments = high_score_posts.join(comments)

# ---------------------------
# OPTIMIZATION BLOCK 2
# ---------------------------
post_votes.cache()
#-----------------------------

user_scores = post_votes.map(lambda x: (x[1][0][0], x[1][1])) \
                        .reduceByKey(lambda a, b: a + b)

tag_scores = post_votes.map(lambda x: (x[1][0][1], x[1][1])) \
                      .reduceByKey(lambda a, b: a + b)

user_comment_scores = post_comments.map(lambda x: (x[1][0][0], x[1][1])) \
                        .reduceByKey(lambda a, b: a + b)


In [10]:
# ---------------------------
# Application 1
# ---------------------------
print(user_scores.take(5))
print(tag_scores.take(5))
print(user_comment_scores.take(5))

[(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
[('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]


[(15, 55974), (0, 22390), (45, 7462), (40, 14924), (1500, 3731)]


In [11]:
# ---------------------------
# Application 2
# ---------------------------
print(user_scores.take(5))
print(tag_scores.take(5))

[(196, 79), (168, 21), (2164, 6), (4096, 7), (2728, 10)]
[('<time-series><mcmc><mixed-model><forecasting>', 10), ('<r><anova><contrasts><sums-of-squares>', 39), ('<econometrics>', 22), ('<mixed-model>', 6), ('<interpretation><moments><parallel-computing>', 22)]


### c) Translate the algorithm for constructing a Top-K user similarity graph from a user-item interaction graph (algorithm from Exercise 2)

In [12]:
# Input edges
edges = [
    ("u1", "i1"), ("u1", "i2"), ("u1", "i3"), ("u1", "i4"),
    ("u2", "i1"), ("u2", "i2"), ("u2", "i5"),
    ("u3", "i2"), ("u3", "i3"), ("u3", "i4"),
    ("u4", "i3"), ("u4", "i5")
]

k = 2 

In [13]:

from itertools import combinations

sc = spark.sparkContext

# Create RDD from input edges
edgesRDD = sc.parallelize(edges)

# Mapper 1: (u, v) => (v, u)  [reverse: item -> user]
# Reducer 1: group by item => (v, [u1, u2, ...])
item_users = (
    edgesRDD
    .map(lambda e: (e[1], e[0]))
    .groupByKey()
)

# Mapper 2: (v, users) => ((u1, u2), 1) for every pair where u1 < u2
# Reducer 2: sum => ((u1, u2), total_common_items)
user_pair_scores = (
    item_users
    .flatMap(lambda x: [
        ((u1, u2), 1)
        for u1, u2 in combinations(sorted(x[1]), 2)
    ])
    .reduceByKey(lambda a, b: a + b)
)

# Mapper 3: ((u1, u2), score) => (u1, (u2, score)) and (u2, (u1, score))
# Reducer 3: for each user keep top-k neighbours by similarity score
top_k_graph = (
    user_pair_scores
    .flatMap(lambda x: [
        (x[0][0], (x[0][1], x[1])),
        (x[0][1], (x[0][0], x[1]))
    ])
    .groupByKey()
    .mapValues(lambda user_score: sorted(user_score, key=lambda c: -c[1])[:k])
)

# Emit ((u, v), score) as final output edges of the top-K similarity graph
result_c = top_k_graph.flatMap(lambda x: [((x[0], v), s) for v, s in x[1]])

print("Top-K Similarity Graph edges:")
for edge, score in sorted(result_c.collect()):
    print(f"  {edge[0]} - {edge[1]}: common_item={score}")


Top-K Similarity Graph edges:


  u1 - u2: common_item=2
  u1 - u3: common_item=3
  u2 - u1: common_item=2
  u2 - u3: common_item=1
  u3 - u1: common_item=3
  u3 - u4: common_item=1
  u4 - u1: common_item=1
  u4 - u3: common_item=1


### d) You are given a dataset of retail transactions, where each transaction consists of a list of items bought together. Using Spark RDD transformations and actions, implement an algorithm to identify the top-K most frequently co-purchased item pairs.

In [14]:
# Input Data
transactions = [
    ["milk", "bread", "eggs", "butter"],
    ["bread", "butter", "jam"],
    ["milk", "bread"],
    ["bread", "eggs"],
    ["milk", "eggs", "yogurt"],
    ["bread", "milk", "butter"],
    ["milk", "bread", "butter", "cheese"],
    ["bread", "eggs", "butter"],
    ["milk", "eggs", "bread"],
    ["bread", "butter", "jam", "honey"],
    ["milk", "yogurt"],
    ["eggs", "bread", "milk", "butter"],
    ["bread", "cheese"],
    ["milk", "bread", "eggs"],
    ["butter", "bread", "jam"],
    ["milk", "bread", "cheese"],
    ["eggs", "milk", "yogurt", "honey"],
    ["bread", "butter", "eggs"],
    ["milk", "bread", "butter", "jam"],
    ["bread", "eggs", "milk"]
]

min_support = 3

In [15]:
from itertools import combinations

TOP_K = 5

transactionsRDD = sc.parallelize(transactions)

top_k_pairs = (
    transactionsRDD
    # Generate all sorted item pairs from each transaction (avoid duplicates by sorting)
    .flatMap(lambda items: [
        tuple(sorted(pair))
        for pair in combinations(items, 2)
    ])
    # Count each pair
    .map(lambda pair: (pair, 1))
    .reduceByKey(lambda a, b: a + b)  # type: ignore
    # Keep only pairs meeting minimum support threshold
    .filter(lambda x: x[1] >= min_support)
    # Collect top-K by descending frequency
    .takeOrdered(TOP_K, key=lambda x: -x[1])  # type: ignore
)

print(f"Top-{TOP_K} most frequently co-purchased item pairs (min_support={min_support}):")
for pair, count in top_k_pairs:
    print(f"  ({pair[0]}, {pair[1]}): {count}")


Top-5 most frequently co-purchased item pairs (min_support=3):
  (bread, butter): 10
  (bread, milk): 10
  (bread, eggs): 8
  (eggs, milk): 7
  (butter, milk): 5


# Stop spark session

In [16]:
spark.stop()

---
## **Your solution for Exercise 5 will consist of:**  
*  This notebook, filled with your solutions and discussions/explanations in the report. 
